In [1]:
"""
Vorbereitung
step 1) erstmal Dateien extrahieren
step 2) manuell in VS txt-format in csv umwandeln (einfach mit rechtsklick + rename)


Anmerkungen zu dfs:

für Temperatur werden einfach Stundendaten verwendet
für Sonnenscheindauer werden 10-Min-Daten verwendet, weil hier sonst keine 15 Min. Auflösung möglich wäre.

Vorgehen: aus 10 Minuten mach 15 Minuten
Aus 15 Minuten mach 4 verschiedene dfs mit Stundenwerten. 

zu klären bleibt, ob der Timeshift mit eingebaut werden soll oder nicht.

"""



'\nVorbereitung\nstep 1) erstmal Dateien extrahieren\nstep 2) manuell in VS txt-format in csv umwandeln (einfach mit rechtsklick + rename)\n\n\nAnmerkungen zu dfs:\n\nfür Temperatur werden einfach Stundendaten verwendet\nfür Sonnenscheindauer werden 10-Min-Daten verwendet, weil hier sonst keine 15 Min. Auflösung möglich wäre.\n\nVorgehen: aus 10 Minuten mach 15 Minuten\nAus 15 Minuten mach 4 verschiedene dfs mit Stundenwerten. \n\nzu klären bleibt, ob der Timeshift mit eingebaut werden soll oder nicht.\n\n'

In [2]:

import pandas as pd
import numpy as np


# name = 'produkt_zehn_min_sd_20000101_20080526_00222.csv' -> war erster Versuch

name = 'Schleswig_2004_1h_solar.csv'
name_tamb = 'Schleswig_2004_1h_tamb.csv'

#welches Jahr soll gefiltert werden?
# #2024
# start = pd.Timestamp('2024-01-01 00:00')
# ende = pd.Timestamp('2024-12-31 23:50')

# 2004
start = pd.Timestamp('2004-01-01 00:00')
ende = pd.Timestamp('2004-12-31 23:50')


# CSV-Datei laden
df = pd.read_csv(name,sep=';')
df_Tamb = pd.read_csv(name_tamb,sep=';')
df_Tamb



,STATIONS_ID,MESS_DATUM,QN_9,TT_TU,RF_TU,eor
0,3032,1949010101,5,2.9,90.0,eor
1,3032,1949010102,5,3.0,89.0,eor
2,3032,1949010103,5,3.0,87.0,eor
3,3032,1949010104,5,2.6,87.0,eor
4,3032,1949010105,5,2.7,88.0,eor
...,...,...,...,...,...,...
665763,3032,2024123119,3,7.2,92.0,eor
665764,3032,2024123120,3,7.1,95.0,eor
665765,3032,2024123121,3,7.3,94.0,eor
665766,3032,2024123122,3,7.6,94.0,eor


In [3]:
#Transition von J/cm^2 [DWD] zu W/m^2 [PVSyst]
# 
# 1W = 1J/s 
# 1m^2 = 100^2 cm^2
# alle Werte sind in 15 Minuten Schritten
# --> 1 J/cm^2 = 100^2[Fläche]/(60[s->min]*15[timestep]*) W/m^2

#energy_to_power_15min = (100^2)/(60*15) # J/cm^2 to W/m^2 [15min]
energy_to_power_1h    = (100^2)/(60^2) # J/cm^2 to W/m^2 [1h] # später addiere ich ja werte und teile sie durch 4 --> braucht nur einen Transition Faktor


In [4]:
# hier muss aufgepasst werden: manche Dateien haben %Y%m%d%H%M als Format, andere haben %Y%m%d%H:%M
df_neu = df

df_neu['MESS_DATUM'] = pd.to_datetime(df['MESS_DATUM'], format  ='%Y%m%d%H:%M') # hier muss aufgepasst werden: manche Dateien haben %Y%m%d%H%M als Format, andere haben %Y%m%d%H:%M
df_neu

df_neu_tamb = df_Tamb
df_neu_tamb['MESS_DATUM'] = pd.to_datetime(df_Tamb['MESS_DATUM'], format='%Y%m%d%H')
df_neu_tamb

,STATIONS_ID,MESS_DATUM,QN_9,TT_TU,RF_TU,eor
0,3032,1949-01-01 01:00:00,5,2.9,90.0,eor
1,3032,1949-01-01 02:00:00,5,3.0,89.0,eor
2,3032,1949-01-01 03:00:00,5,3.0,87.0,eor
3,3032,1949-01-01 04:00:00,5,2.6,87.0,eor
4,3032,1949-01-01 05:00:00,5,2.7,88.0,eor
...,...,...,...,...,...,...
665763,3032,2024-12-31 19:00:00,3,7.2,92.0,eor
665764,3032,2024-12-31 20:00:00,3,7.1,95.0,eor
665765,3032,2024-12-31 21:00:00,3,7.3,94.0,eor
665766,3032,2024-12-31 22:00:00,3,7.6,94.0,eor


In [5]:

# Filtere den DataFrame im gewünschten Zeitraum
df_neu = df_neu[(df_neu['MESS_DATUM'] >= start) & (df_neu['MESS_DATUM'] <= ende)].reset_index(drop=True) 
df_neu_tamb = df_neu_tamb[(df_neu_tamb['MESS_DATUM'] >= start) & (df_neu_tamb['MESS_DATUM'] <= ende)].reset_index(drop=True)

df_neu



,STATIONS_ID,MESS_DATUM,QN_592,ATMO_LBERG,FD_LBERG,FG_LBERG,SD_LBERG,ZENIT,MESS_DATUM_WOZ,eor
0,3032,2004-01-01 00:29:00,1,-999,0.0,0.0,0,147.57,2004010101:00,eor
1,3032,2004-01-01 01:29:00,1,-999,0.0,0.0,0,143.94,2004010102:00,eor
2,3032,2004-01-01 02:29:00,1,-999,0.0,0.0,0,137.69,2004010103:00,eor
3,3032,2004-01-01 03:29:00,1,-999,0.0,0.0,0,129.95,2004010104:00,eor
4,3032,2004-01-01 04:29:00,1,-999,0.0,0.0,0,121.52,2004010105:00,eor
...,...,...,...,...,...,...,...,...,...,...
8779,3032,2004-12-31 19:29:00,1,-999,0.0,0.0,0,121.50,2004123120:00,eor
8780,3032,2004-12-31 20:29:00,1,-999,0.0,0.0,0,129.92,2004123121:00,eor
8781,3032,2004-12-31 21:29:00,1,-999,0.0,0.0,0,137.65,2004123122:00,eor
8782,3032,2004-12-31 22:29:00,1,-999,0.0,0.0,0,143.89,2004123123:00,eor


In [6]:

def timeshift_UTC_to_WOZ(messdaten):
    '''
    description:
    funktioniert für Zeitreihen mit beliebiger länge
    verschiebt die Zeitreihe im pd-df-Format von UTC auf WOZ Zeit
    checkout https://www.pveducation.org/pvcdrom/properties-of-sunlight/solar-time for equation explenation (I used Equation of Time (EoT) for timeshift_UTC_to_WOZ )

    input:
    pd df mit Spalte 'MESS_DATUM'
    output:
    gibt Spalte mit korrigierten Zeitstempeln zurück
    '''

    # initialisieren der np arrays --> kann später auch gekürzt werden, wenn eh Schleife genutzt wird, müssen einzelne Werte nicht gespeichert werden
    # B = np.zeros(len(messdaten))
    # timeshift_UTC_to_WOZ = np.zeros(len(messdaten))
    WOZ = messdaten['MESS_DATUM']

    for i in range(len(messdaten)):  
        # hier wird Formel für shift von UTC zu WOZ umgesetzt --> der WOZ-df hat als Messzeitpunkt Einträge immer die Zeitlichen Einträge, verschoben um die Korrektur, die sich aus der oben visualisierten Kurve ergibt
        
        today = messdaten['MESS_DATUM'][i]
        d = today - start 
        d = d.days
        B = 360/365*(d - 81)
        timeshift_UTC_to_WOZ = round(9.87* np.sin(  np.deg2rad(2*B)  ) -   7.53 * np.cos(  np.deg2rad(B)  )  - 1.5 * np.sin( np.deg2rad(B) ))
        WOZ[i] = today + pd.to_timedelta(timeshift_UTC_to_WOZ, unit='m')
    return(WOZ)


WOZ = timeshift_UTC_to_WOZ(df_neu)
WOZ

# hier sieht man, warum man keine for schleife benutzen sollte... das dauert einfach ewig

C:\Users\CasparGross\AppData\Local\Temp\ipykernel_34520\3311359829.py:27: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  WOZ[i] = today + pd.to_timedelta(timeshift_UTC_to_WOZ, unit='m')
C:\Users\CasparGross\AppData\Local\Temp\ipykernel_34520\3311359829.py:27: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  WOZ[i] = today + pd.to_timedelta(timeshift_UTC_to_WOZ, unit='m')
C:\Users\CasparGross\AppData\Local\Temp\ipykernel_34520\3311359829.py:27: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_gu

0      2004-01-01 00:26:00
1      2004-01-01 01:26:00
2      2004-01-01 02:26:00
3      2004-01-01 03:26:00
4      2004-01-01 04:26:00
               ...        
8779   2004-12-31 19:26:00
8780   2004-12-31 20:26:00
8781   2004-12-31 21:26:00
8782   2004-12-31 22:26:00
8783   2004-12-31 23:26:00
Name: MESS_DATUM, Length: 8784, dtype: datetime64[ns]

In [ ]:
df_neu_relevant = df_neu[['MESS_DATUM']]

# hier  auf WOZ-Timestamps wechseln und in Format von PV-Sys speichern (für alles eigene Spalten)

df_neu_relevant['Year'] = WOZ.dt.year
df_neu_relevant['Month'] = WOZ.dt.month
df_neu_relevant['Day'] = WOZ.dt.day
df_neu_relevant['Hour'] = WOZ.dt.hour
df_neu_relevant['Minute'] = WOZ.dt.minute

df_neu_relevant['GHI'] = df_neu['FG_LBERG'] * energy_to_power_1h
df_neu_relevant['DHI'] = df_neu['FD_LBERG'] * energy_to_power_1h
# Temperatur hinzufügen
df_neu_relevant['Tamb'] = df_neu_tamb['TT_TU']

# da ich aus 1h Werten 1h Werte mache, brauche ich hier keine weiteren Berechnungen (nur Timeshift von oben)


df_neu_relevant.drop(columns=['MESS_DATUM'], inplace=True)

df_neu_relevant

#   -->  Ziel Format: ;;;;;W/m2;W/m2;deg.C

C:\Users\CasparGross\AppData\Local\Temp\ipykernel_34520\1050563773.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_neu_relevant['Year'] = WOZ.dt.year
C:\Users\CasparGross\AppData\Local\Temp\ipykernel_34520\1050563773.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_neu_relevant['Month'] = WOZ.dt.month
C:\Users\CasparGross\AppData\Local\Temp\ipykernel_34520\1050563773.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_

,Year,Month,Day,Hour,Minute,GHI,DHI,Tamb
0,2004,1,1,0,26,0.0,0.0,-0.9
1,2004,1,1,1,26,0.0,0.0,-0.9
2,2004,1,1,2,26,0.0,0.0,-1.1
3,2004,1,1,3,26,0.0,0.0,-1.3
4,2004,1,1,4,26,0.0,0.0,-1.2
...,...,...,...,...,...,...,...,...
8779,2004,12,31,19,26,0.0,0.0,4.5
8780,2004,12,31,20,26,0.0,0.0,4.0
8781,2004,12,31,21,26,0.0,0.0,4.4
8782,2004,12,31,22,26,0.0,0.0,4.4


In [8]:

new_row = ['', '', '', '', '','W/m2','W/m2','deg.C']   

# einheiten einfügen bei CSV -> entkommentieren der beiden Zeilen unten
# #df_15 = pd.concat([pd.DataFrame([new_row], columns=df_15.columns), df_15], ignore_index=True) # nur einmal einfügen
# Einheiten-Spalte nur einmal einfügen:
df_neu_relevant = pd.concat([pd.DataFrame([new_row], columns=df_neu_relevant.columns), df_neu_relevant], ignore_index=True) 


df_neu_relevant

,Year,Month,Day,Hour,Minute,GHI,DHI,Tamb
0,,,,,,W/m2,W/m2,deg.C
1,2004,1,1,0,26,0.0,0.0,-0.9
2,2004,1,1,1,26,0.0,0.0,-0.9
3,2004,1,1,2,26,0.0,0.0,-1.1
4,2004,1,1,3,26,0.0,0.0,-1.3
...,...,...,...,...,...,...,...,...
8780,2004,12,31,19,26,0.0,0.0,4.5
8781,2004,12,31,20,26,0.0,0.0,4.0
8782,2004,12,31,21,26,0.0,0.0,4.4
8783,2004,12,31,22,26,0.0,0.0,4.4


In [ ]:
#achtung! name muss noch im cache sein!

# name_new_1 = name.replace('.csv', '_2024_1h_1_for_pv_syst.csv')
from ntpath import join

stationsID = 'Potsdam_station_03032_neu_'
start_string = start.strftime("%Y-%m-%d")
ende_string = ende.strftime("%Y-%m-%d") 
timestep = '1h_'
csv_ening = 'for_pv_syst.csv'
name_new_1 = ''.join([stationsID,start_string,ende_string,timestep,'_1_',csv_ening])

# nur ausführen wenn Datei ready ist, sonst wird bestehende Datei überschrieben 
df_neu_relevant.to_csv(name_new_1, sep=';', index=False, header=['Year','Month','Day','Hour','Minute','GHI','DHI','Tamb'])


#   diese Header händisch hinzufügen

#Site; Potsdam_for_3081_HAGE;;;;;;;
#Country; germany;;;;;;;
#Data Source; DWD_station_03987;;;;;;;
#Time step; 1h;;;;;;;
#Year; 2024;;;;;;;
#Latitude; 52.3812;;;;;;;
#Longitude; 13.0622;;;;;;;
#Altitude; 81;;;;;;;
#Time Zone; 1  ;;;;;;;

# --> muss aber individuell je nach Metadaten angepasst werden (könnte man aber in Zukunft automatisieren)

# Zielformat
# Year;Month;Day;Hour;Minute;GHI;Tamb;
# ;;;;;W/m2;deg.C;

In [10]:
"""
ToDo:

# location checken
# Windgeschwindigkeit u Temp ergänzen --> müsste aus Metadaten hervorgehen, wo die zu finden sind.

# Simulieren mit Lima

# in Zukunft Interpolieren, wenn Werte fehlen!!! -> hat jetzt durch Zufall auch ohne funktioniert, aber kann man sich nicht 100%ig drauf verlassen

# Wetterstationen filtern später

UTF-Zeitformat anpassen --> bisher ist noch nicht der Switch zwischen UTF + 1 und local Time integriert
wenn ich das aber richtig aus dr Doku lese, dann reicht es am Anfang #Time Reference; 1  zu ergänzen --> das müsste für PV-Syst dann klar stellen, dass es selbst den Shift von UTF 1 zu local time macht

https://www.pvsyst.com/help/meteo-database/notes-on-meteo/meteonote7-time-shift-in-meteo-files.html#time-shift-correction-when-importing-data 
If there is a time shift varying along the year:

With the general ASCII importing tool, consider using the DST correction (option "Summer Time > Apply").
Otherwise there is no practical way, except modifying the dates directly in the source file.
If the file is written in UT (Universal Time):

With the general ASCII importing tool, check the box "Universal time".
With PVsyst standard format, you can add a line #Time Reference with the value UT

"""

'\nToDo:\n\n# location checken\n# Windgeschwindigkeit u Temp ergänzen --> müsste aus Metadaten hervorgehen, wo die zu finden sind.\n\n# Simulieren mit Lima\n\n# in Zukunft Interpolieren, wenn Werte fehlen!!! -> hat jetzt durch Zufall auch ohne funktioniert, aber kann man sich nicht 100%ig drauf verlassen\n\n# Wetterstationen filtern später\n\nUTF-Zeitformat anpassen --> bisher ist noch nicht der Switch zwischen UTF + 1 und local Time integriert\nwenn ich das aber richtig aus dr Doku lese, dann reicht es am Anfang #Time Reference; 1  zu ergänzen --> das müsste für PV-Syst dann klar stellen, dass es selbst den Shift von UTF 1 zu local time macht\n\nhttps://www.pvsyst.com/help/meteo-database/notes-on-meteo/meteonote7-time-shift-in-meteo-files.html#time-shift-correction-when-importing-data \nIf there is a time shift varying along the year:\n\nWith the general ASCII importing tool, consider using the DST correction (option "Summer Time > Apply").\nOtherwise there is no practical way, except